# Notebook 9 — Optimizador de Portafolio (Markowitz) → MongoDB
### Ernesto Investing AI · iDeSo · UNMSM

**Objetivo:** leer los precios históricos reales de los 5 tickers mineros desde `precios_ohlcv`, calcular sus retornos y estadísticas de riesgo/retorno, y optimizar 3 perfiles de portafolio (Conservador, Moderado, Agresivo) usando la **Teoría Moderna de Portafolio de Markowitz** (Mean-Variance Optimization) con `scipy.optimize`.

Los resultados alimentan la pestaña **"Estrategias"** del frontend (controles: Capital Inicial, Horizonte Temporal, Nivel de Riesgo), reemplazando los datos simulados (VOO, BTC, Bonos, QQQ) por una asignación real entre los 5 activos mineros del proyecto.

**Tickers del proyecto:**
- `FSM` — Fortuna Silver Mines Inc.
- `VOLCABC1.LM` — Volcan Compañía Minera S.A.A.
- `ABX.TO` — Barrick Gold Corporation
- `BVN` — Compañía de Minas Buenaventura
- `BHP` — BHP Group Limited

**Metodología:**
- Sin venta en corto (`weights >= 0`), sin apalancamiento (`sum(weights) = 1`).
- **Conservador** → varianza mínima global.
- **Moderado** → máximo Sharpe Ratio (tasa libre de riesgo = 4.5% anual).
- **Agresivo** → máximo retorno esperado, con tope de 50% por activo (evita concentración degenerada).
- Se repite para 4 horizontes temporales (3 Meses, 6 Meses, 1 Año, 3 Años) → **12 combinaciones**.

**Persistencia:** MongoDB Atlas (`db='spbi'`), colección nueva `estrategias`.

**Prerequisito:** definir el secreto `MONGO_URI` en Colab (ícono de llave 🔑 en el panel izquierdo) con la cadena de conexión a tu clúster de MongoDB Atlas.

## Paso 1 — Instalar librerias necesarias

In [ ]:
# Paso 1 — Instalar librerias necesarias
!pip install "pymongo[srv]" scipy --quiet
print("Librerias instaladas correctamente")

## Paso 2 — Conectar a MongoDB Atlas usando el secreto MONGO_URI de Colab

In [ ]:
# Paso 2 — Conectar a MongoDB Atlas usando el secreto MONGO_URI de Colab
from google.colab import userdata
from pymongo import MongoClient

MONGO_URI = userdata.get("MONGO_URI")  # Debe estar definido en Secrets (icono de llave)
client = MongoClient(MONGO_URI)
db = client["spbi"]

# Verificar la conexion con un ping
client.admin.command("ping")
print("Conexion a MongoDB Atlas establecida correctamente")
print("Base de datos activa:", db.name)

## Paso 3 — Definir los 5 tickers del proyecto y fijar la semilla de reproducibilidad

In [ ]:
# Paso 3 — Definir los 5 tickers mineros del proyecto y fijar semilla
import numpy as np
import pandas as pd
import random
from datetime import datetime
from scipy.optimize import minimize

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

TICKERS = {
    "FSM": "Fortuna Silver Mines Inc.",
    "VOLCABC1.LM": "Volcan Compania Minera S.A.A.",
    "ABX.TO": "Barrick Gold Corporation",
    "BVN": "Compania de Minas Buenaventura",
    "BHP": "BHP Group Limited",
}
LISTA_TICKERS = list(TICKERS.keys())
N_ACTIVOS = len(LISTA_TICKERS)

# Tasa libre de riesgo anual (aproximacion a T-Bills USD), usada en el perfil Moderado
TASA_LIBRE_RIESGO = 0.045

# Tope maximo de concentracion por activo, usado en el perfil Agresivo
TOPE_CONCENTRACION = 0.50

print("Tickers del proyecto:")
for ticker, nombre in TICKERS.items():
    print(f"  {ticker}: {nombre}")
print()
print(f"SEED={SEED} | tasa libre de riesgo={TASA_LIBRE_RIESGO:.1%} | tope de concentracion={TOPE_CONCENTRACION:.0%}")

## Paso 4 — Cargar precios desde MongoDB y calcular retornos diarios

Se lee la coleccion `precios_ohlcv` para cada ticker, se ordena por fecha y se calcula el retorno diario como el cambio porcentual (`pct_change`) del precio de cierre.

In [ ]:
# Paso 4 — Cargar precios OHLCV desde MongoDB y calcular retornos diarios
def cargar_precios(ticker):
    """Lee los documentos de precios_ohlcv para un ticker y devuelve un DataFrame ordenado por fecha."""
    docs = list(db["precios_ohlcv"].find({"ticker": ticker}, {"_id": 0, "fecha": 1, "close": 1}).sort("fecha", 1))
    if not docs:
        return pd.DataFrame(columns=["fecha", "close"])
    df = pd.DataFrame(docs)
    df["fecha"] = pd.to_datetime(df["fecha"])
    df = df.dropna(subset=["close"]).sort_values("fecha").reset_index(drop=True)
    return df


# Se cargan los 5 tickers y se calcula el retorno diario (pct_change del cierre)
precios_por_ticker = {}
for ticker in LISTA_TICKERS:
    df = cargar_precios(ticker)
    df["retorno"] = df["close"].pct_change()
    precios_por_ticker[ticker] = df
    print(f"  [{ticker}] {len(df)} precios cargados | {df['retorno'].notna().sum()} retornos diarios validos")

# Se construye una matriz de retornos alineada por fecha (interseccion de fechas comunes a los 5 tickers)
df_retornos_completo = None
for ticker, df in precios_por_ticker.items():
    serie = df.set_index("fecha")["retorno"].rename(ticker)
    df_retornos_completo = serie.to_frame() if df_retornos_completo is None else df_retornos_completo.join(serie, how="inner")

df_retornos_completo = df_retornos_completo.dropna()
print()
print(f"Matriz de retornos diarios: {df_retornos_completo.shape[0]} dias en comun x {df_retornos_completo.shape[1]} tickers")
print(f"Rango de fechas: {df_retornos_completo.index.min().date()} a {df_retornos_completo.index.max().date()}")

## Paso 5 — Definir los horizontes temporales del frontend

El control "Horizonte Temporal" del frontend recorta la ventana de datos historicos usada para calcular retornos y covarianza:

| Horizonte | Ventana de dias de trading |
|---|---|
| 3 Meses (`3m`) | ultimos 63 dias |
| 6 Meses (`6m`) | ultimos 126 dias |
| 1 Año (`1y`) | ultimos 252 dias (o todo el historico si es menor) |
| 3 Años (`3y`) | todo el historico disponible (el proyecto solo tiene ~1 año de datos, asi que en la practica equivale a "todo") |

In [ ]:
# Paso 5 — Definir los horizontes temporales y la funcion de recorte
HORIZONTES = {
    "3m": 63,
    "6m": 126,
    "1y": 252,
    "3y": None,  # None = usar todo el historico disponible
}


def recortar_por_horizonte(df_retornos, dias):
    """Devuelve los ultimos N dias de la matriz de retornos, o todo el historico si dias es None."""
    if dias is None or dias >= len(df_retornos):
        return df_retornos
    return df_retornos.tail(dias)


for horizonte, dias in HORIZONTES.items():
    ventana = recortar_por_horizonte(df_retornos_completo, dias)
    print(f"  {horizonte:3s} -> {len(ventana)} dias de trading utilizados")

## Paso 6 — Calcular estadisticas de portafolio (retorno, covarianza, volatilidad)

- **Retorno esperado anualizado** por ticker: retorno diario promedio x 252.
- **Matriz de covarianza anualizada**: covarianza diaria x 252.
- **Volatilidad individual anualizada** por ticker: usada para la columna "Riesgo".

In [ ]:
# Paso 6 — Funcion de estadisticas de portafolio (retorno, covarianza, volatilidad anualizados)
DIAS_TRADING_ANIO = 252


def calcular_estadisticas(df_retornos):
    """Calcula retorno esperado anual, matriz de covarianza anual y volatilidad anual por ticker."""
    retorno_anual = df_retornos.mean() * DIAS_TRADING_ANIO
    covarianza_anual = df_retornos.cov() * DIAS_TRADING_ANIO
    volatilidad_anual = df_retornos.std() * np.sqrt(DIAS_TRADING_ANIO)
    return retorno_anual, covarianza_anual, volatilidad_anual


# Vista previa con el horizonte de 1 año
mu_preview, cov_preview, vol_preview = calcular_estadisticas(recortar_por_horizonte(df_retornos_completo, HORIZONTES["1y"]))
print("Retorno esperado anualizado (horizonte 1y):")
print(mu_preview.round(4))
print()
print("Volatilidad anualizada (horizonte 1y):")
print(vol_preview.round(4))

## Paso 7 — Funciones de optimizacion de portafolio (SLSQP)

Se definen las funciones de retorno y volatilidad del portafolio, y los 3 optimizadores (uno por perfil de riesgo), todos con `weights >= 0` (sin venta en corto) y `sum(weights) == 1` (sin apalancamiento). Si `scipy.optimize` no converge, se usa un fallback de pesos iguales (1/5 cada ticker).

In [ ]:
# Paso 7 — Funciones de retorno/volatilidad del portafolio y optimizadores por perfil de riesgo
def retorno_portafolio(pesos, mu):
    return float(np.dot(pesos, mu.values))


def volatilidad_portafolio(pesos, cov):
    return float(np.sqrt(pesos @ cov.values @ pesos))


def _pesos_iguales():
    return np.repeat(1.0 / N_ACTIVOS, N_ACTIVOS)


def _restriccion_suma_uno():
    return [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]


def optimizar_varianza_minima(mu, cov):
    """Perfil CONSERVADOR: minimiza la volatilidad del portafolio (varianza minima global)."""
    x0 = _pesos_iguales()
    bounds = [(0.0, 1.0)] * N_ACTIVOS
    res = minimize(volatilidad_portafolio, x0, args=(cov,), method="SLSQP",
                    bounds=bounds, constraints=_restriccion_suma_uno())
    return (res.x, True) if res.success else (_pesos_iguales(), False)


def optimizar_max_sharpe(mu, cov, tasa_libre_riesgo=TASA_LIBRE_RIESGO):
    """Perfil MODERADO: maximiza el Sharpe Ratio (portafolio de tangencia)."""
    x0 = _pesos_iguales()
    bounds = [(0.0, 1.0)] * N_ACTIVOS

    def sharpe_negativo(w):
        vol = volatilidad_portafolio(w, cov)
        if vol == 0:
            return 0.0
        return -(retorno_portafolio(w, mu) - tasa_libre_riesgo) / vol

    res = minimize(sharpe_negativo, x0, method="SLSQP",
                    bounds=bounds, constraints=_restriccion_suma_uno())
    return (res.x, True) if res.success else (_pesos_iguales(), False)


def optimizar_max_retorno_topado(mu, cov, tope=TOPE_CONCENTRACION):
    """Perfil AGRESIVO: maximiza el retorno esperado, con tope de concentracion por activo."""
    x0 = _pesos_iguales()
    bounds = [(0.0, tope)] * N_ACTIVOS

    def retorno_negativo(w):
        return -retorno_portafolio(w, mu)

    res = minimize(retorno_negativo, x0, method="SLSQP",
                    bounds=bounds, constraints=_restriccion_suma_uno())
    return (res.x, True) if res.success else (_pesos_iguales(), False)


OPTIMIZADORES = {
    "conservador": lambda mu, cov: optimizar_varianza_minima(mu, cov),
    "moderado": lambda mu, cov: optimizar_max_sharpe(mu, cov),
    "agresivo": lambda mu, cov: optimizar_max_retorno_topado(mu, cov),
}

print("Optimizadores definidos: conservador (varianza minima), moderado (max Sharpe), agresivo (max retorno topado)")

## Paso 8 — Construir la frontera eficiente

Se calculan ~30 portafolios optimos variando el retorno objetivo entre el minimo y el maximo retorno individual, minimizando la volatilidad para cada nivel de retorno (misma logica que Markowitz, con restricciones adicionales de retorno objetivo).

In [ ]:
# Paso 8 — Construccion de la frontera eficiente (~30 puntos retorno vs volatilidad)
def calcular_frontera_eficiente(mu, cov, n_puntos=30):
    """Minimiza la volatilidad del portafolio para una grilla de retornos objetivo entre min(mu) y max(mu)."""
    retorno_min, retorno_max = mu.min(), mu.max()
    objetivos = np.linspace(retorno_min, retorno_max, n_puntos)
    bounds = [(0.0, 1.0)] * N_ACTIVOS
    x0 = _pesos_iguales()

    puntos = []
    for objetivo in objetivos:
        restricciones = [
            {"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
            {"type": "eq", "fun": lambda w, obj=objetivo: retorno_portafolio(w, mu) - obj},
        ]
        res = minimize(volatilidad_portafolio, x0, args=(cov,), method="SLSQP",
                        bounds=bounds, constraints=restricciones)
        if res.success:
            puntos.append({"retorno": float(objetivo), "volatilidad": float(res.fun)})

    return puntos


# Vista previa de la frontera con el horizonte de 1 año
frontera_preview = calcular_frontera_eficiente(mu_preview, cov_preview, n_puntos=30)
print(f"Frontera eficiente calculada: {len(frontera_preview)} puntos (horizonte 1y)")
print("Primeros 3 puntos:", [ (round(p['retorno'],3), round(p['volatilidad'],3)) for p in frontera_preview[:3] ])

## Paso 9 — Clasificar el riesgo individual de cada ticker por terciles de volatilidad

Para la columna "Riesgo" de la tabla del frontend, cada ticker se clasifica como `Bajo`, `Medio` o `Alto` segun el tercil de volatilidad anualizada en el que caiga, dentro del horizonte evaluado.

In [ ]:
# Paso 9 — Clasificacion de riesgo individual por terciles de volatilidad
def clasificar_riesgo_por_terciles(volatilidad_anual):
    """Devuelve un dict {ticker: 'Bajo'|'Medio'|'Alto'} segun terciles de volatilidad anualizada."""
    orden = volatilidad_anual.sort_values()
    etiquetas = {}
    n = len(orden)
    for i, (ticker, _) in enumerate(orden.items()):
        posicion = i / n  # 0.0 a <1.0
        if posicion < 1/3:
            etiquetas[ticker] = "Bajo"
        elif posicion < 2/3:
            etiquetas[ticker] = "Medio"
        else:
            etiquetas[ticker] = "Alto"
    return etiquetas


riesgo_preview = clasificar_riesgo_por_terciles(vol_preview)
print("Clasificacion de riesgo por terciles (horizonte 1y):")
for ticker, nivel in riesgo_preview.items():
    print(f"  {ticker:12s}: {nivel}")

## Paso 10 — Construir el documento de estrategia por combinacion (perfil x horizonte)

Se ensambla el documento con el formato exacto requerido por el frontend (`Activo`, `Tipo`, `Asignacion %`, `Riesgo`, `Rendimiento Est.` para cada uno de los 5 tickers, incluso con asignacion 0%).

In [ ]:
# Paso 10 — Funcion que arma el documento final de una combinacion perfil x horizonte
def construir_documento_estrategia(perfil, horizonte, pesos, exito_optimizacion, mu, cov, vol, frontera):
    retorno_esperado = retorno_portafolio(pesos, mu)
    volatilidad = volatilidad_portafolio(pesos, cov)
    sharpe = (retorno_esperado - TASA_LIBRE_RIESGO) / volatilidad if volatilidad > 0 else 0.0
    riesgo_por_ticker = clasificar_riesgo_por_terciles(vol)

    activos = []
    for i, ticker in enumerate(LISTA_TICKERS):
        activos.append({
            "ticker": ticker,
            "tipo": "Accion Minera",
            "asignacion_pct": round(float(pesos[i]) * 100, 2),
            "riesgo": riesgo_por_ticker[ticker],
            "rendimiento_estimado_pct": round(float(mu[ticker]) * 100, 2),
        })

    documento = {
        "perfil_riesgo": perfil,
        "horizonte": horizonte,
        "retorno_esperado_anual": round(float(retorno_esperado), 4),
        "volatilidad_anual": round(float(volatilidad), 4),
        "sharpe_ratio": round(float(sharpe), 4),
        "activos": activos,
        "frontera_eficiente": frontera,
        "optimizacion_exitosa": bool(exito_optimizacion),
        "fecha_calculo": datetime.now(),
        "created_at": datetime.now(),
    }
    return documento


print("Funcion construir_documento_estrategia definida")

## Paso 11 — Visualizacion de la frontera eficiente por horizonte

Para cada uno de los 4 horizontes se grafica la frontera eficiente (retorno vs volatilidad) con los 3 perfiles de riesgo marcados sobre la curva.

In [ ]:
# Paso 11 — Grafico de la frontera eficiente con los 3 perfiles marcados, por horizonte
import matplotlib.pyplot as plt

COLORES_PERFIL = {"conservador": "#06b6d4", "moderado": "#2563eb", "agresivo": "#ef4444"}


def graficar_frontera(horizonte, frontera, puntos_perfiles):
    """puntos_perfiles: dict {perfil: (retorno, volatilidad)}"""
    xs = [p["volatilidad"] for p in frontera]
    ys = [p["retorno"] for p in frontera]

    plt.figure(figsize=(7, 5))
    plt.plot(xs, ys, color="#94a3b8", linewidth=2, label="Frontera eficiente")

    for perfil, (ret, vol) in puntos_perfiles.items():
        plt.scatter(vol, ret, color=COLORES_PERFIL[perfil], s=90, zorder=5, label=perfil.capitalize())

    plt.title(f"Frontera eficiente — horizonte {horizonte}")
    plt.xlabel("Volatilidad anualizada")
    plt.ylabel("Retorno esperado anualizado")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


print("Funcion graficar_frontera definida")

## Paso 12 — Ejecutar la optimizacion para las 12 combinaciones y guardar en MongoDB

Para cada uno de los 4 horizontes y los 3 perfiles de riesgo se recalculan las estadisticas, se optimiza el portafolio, se construye la frontera eficiente, se grafica y se guarda el documento en la coleccion `estrategias` (se eliminan documentos previos de la misma combinacion antes de insertar, igual que en los notebooks anteriores).

In [ ]:
# Paso 12 — Pipeline completo: 4 horizontes x 3 perfiles = 12 combinaciones
resultados = []  # se usa luego para el resumen final y la exportacion a JSON

print("=" * 60)
print("  OPTIMIZADOR DE PORTAFOLIO — 12 COMBINACIONES")
print("=" * 60)

for horizonte, dias in HORIZONTES.items():
    ventana = recortar_por_horizonte(df_retornos_completo, dias)
    mu, cov, vol = calcular_estadisticas(ventana)
    frontera = calcular_frontera_eficiente(mu, cov, n_puntos=30)

    print(f"\nHorizonte: {horizonte} ({len(ventana)} dias de trading)")

    puntos_perfiles = {}
    documentos_horizonte = []

    for perfil, optimizador in OPTIMIZADORES.items():
        pesos, exito = optimizador(mu, cov)
        doc = construir_documento_estrategia(perfil, horizonte, pesos, exito, mu, cov, vol, frontera)
        documentos_horizonte.append(doc)
        resultados.append(doc)

        puntos_perfiles[perfil] = (doc["retorno_esperado_anual"], doc["volatilidad_anual"])
        estado = "OK" if exito else "FALLBACK (pesos iguales)"
        print(f"  [{perfil:11s}] retorno={doc['retorno_esperado_anual']:.2%} | "
              f"vol={doc['volatilidad_anual']:.2%} | sharpe={doc['sharpe_ratio']:.2f} | {estado}")

    # Grafico de la frontera eficiente para este horizonte
    graficar_frontera(horizonte, frontera, puntos_perfiles)

    # Guardado en MongoDB: se borran documentos previos de la misma combinacion y se insertan los nuevos
    for doc in documentos_horizonte:
        db["estrategias"].delete_many({"perfil_riesgo": doc["perfil_riesgo"], "horizonte": doc["horizonte"]})
        db["estrategias"].insert_one(doc)

print()
print("=" * 60)
print(f"  Pipeline completo: {len(resultados)} documentos guardados en 'estrategias'")
print("=" * 60)

## Paso 13 — Verificacion en MongoDB

In [ ]:
# Paso 13 — Celda de verificacion final
total_docs = db["estrategias"].count_documents({})
print(f"Total de documentos en la coleccion 'estrategias': {total_docs}")
print()

print("Combinaciones almacenadas:")
for doc in db["estrategias"].find({}, {"_id": 0, "perfil_riesgo": 1, "horizonte": 1,
                                        "retorno_esperado_anual": 1, "volatilidad_anual": 1,
                                        "sharpe_ratio": 1, "optimizacion_exitosa": 1}).sort([("horizonte", 1), ("perfil_riesgo", 1)]):
    print(f"  {doc['horizonte']:3s} | {doc['perfil_riesgo']:11s} | "
          f"retorno={doc['retorno_esperado_anual']:.2%} | vol={doc['volatilidad_anual']:.2%} | "
          f"sharpe={doc['sharpe_ratio']:.2f} | exitosa={doc['optimizacion_exitosa']}")

print()
muestra = db["estrategias"].find_one({})
print("Documento de muestra (campos principales):")
print({k: v for k, v in muestra.items() if k not in ("activos", "frontera_eficiente")})

## Paso 14 — Exportar `datos_estrategias.json`

Se exportan los 12 documentos a un archivo JSON local (mismo patron de exportacion que los notebooks anteriores del proyecto), convirtiendo los campos `datetime` a texto ISO para que el archivo sea JSON valido.

In [ ]:
# Paso 14 — Exportar los 12 documentos de estrategias a datos_estrategias.json
import json

def serializar_documento(doc):
    """Convierte un documento de MongoDB a un dict JSON-serializable (datetime -> isoformato)."""
    limpio = {k: v for k, v in doc.items() if k != "_id"}
    limpio["fecha_calculo"] = limpio["fecha_calculo"].isoformat()
    limpio["created_at"] = limpio["created_at"].isoformat()
    return limpio

documentos_export = [serializar_documento(doc) for doc in db["estrategias"].find({}).sort([("horizonte", 1), ("perfil_riesgo", 1)])]

with open("datos_estrategias.json", "w", encoding="utf-8") as f:
    json.dump(documentos_export, f, ensure_ascii=False, indent=2)

print(f"Archivo datos_estrategias.json exportado con {len(documentos_export)} combinaciones")

# Descarga automatica en Google Colab
try:
    from google.colab import files
    files.download("datos_estrategias.json")
except ImportError:
    print("(Descarga automatica disponible solo en Google Colab; el archivo quedo guardado localmente)")

## Paso 15 — Resumen final: pesos de las 12 combinaciones

In [ ]:
# Paso 15 — Tabla resumen legible con los pesos de las 12 combinaciones
print("RESUMEN FINAL — ASIGNACION DE ACTIVOS POR COMBINACION")
print("=" * 100)
encabezado = f"{'Horizonte':10s} {'Perfil':12s}" + "".join(f"{t:>14s}" for t in LISTA_TICKERS) + f"{'Retorno':>10s} {'Vol.':>8s}"
print(encabezado)
print("-" * 100)

for doc in sorted(resultados, key=lambda d: (list(HORIZONTES.keys()).index(d["horizonte"]), d["perfil_riesgo"])):
    pesos_txt = "".join(f"{a['asignacion_pct']:>13.1f}%" for a in doc["activos"])
    fila = (f"{doc['horizonte']:10s} {doc['perfil_riesgo']:12s}{pesos_txt}"
            f"{doc['retorno_esperado_anual']:>9.1%} {doc['volatilidad_anual']:>7.1%}")
    print(fila)

print("=" * 100)
print(f"Total: {len(resultados)} combinaciones (4 horizontes x 3 perfiles de riesgo)")

## Resultado

La coleccion `estrategias` contiene 12 portafolios optimos reales (3 perfiles de riesgo x 4 horizontes temporales), calculados con la Teoria Moderna de Portafolio de Markowitz sobre los 5 tickers mineros del proyecto, sin venta en corto y sin apalancamiento.

Cada documento incluye la asignacion porcentual por activo, su nivel de riesgo (terciles de volatilidad), el rendimiento estimado, el Sharpe Ratio del portafolio y ~30 puntos de la frontera eficiente para graficar en el frontend.

**Pipeline:** MongoDB (`precios_ohlcv`) → retornos diarios → estadisticas anualizadas → optimizacion SLSQP (3 perfiles x 4 horizontes) → frontera eficiente → **MongoDB (`estrategias`)** + `datos_estrategias.json` ✓

Estos datos reemplazan la simulacion estatica de la pestaña "Estrategias" del frontend (VOO, BTC, Bonos, QQQ) por una asignacion real entre `FSM`, `VOLCABC1.LM`, `ABX.TO`, `BVN` y `BHP`.